In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)


In [ ]:
from pathlib import Path
import csv

from src.drive_service.logging_utils import setup_logging
from src.pipeline_paths import build_pipelines_paths
from src.extract_days_from_text_raw import build_days_from_text_dir


In [ ]:
 

root = "1FUosjKncLt18JzojmX8tKQm1nbgPI133"
paths = build_pipelines_paths(root)

if not paths.text_extraction_output.exists():
    raise FileNotFoundError(
        f"Text extraction output directory does not exist: {paths.text_extraction_output}"
    )

paths.text_extraction_output, paths.parsing_output


In [ ]:
verbose = True
text_glob = "*.txt"
out_name = "days.csv"
report_json = paths.parsing_output / "extract_days_from_text_raw.report.json"

max_no_days_files = 80
max_no_days_lines = 8

setup_logging(verbose)

report = build_days_from_text_dir(
    input_dir=str(paths.text_extraction_output),
    out_dir=str(paths.parsing_output),
    out_name=out_name,
    report_json=str(report_json),
    text_glob=text_glob,
    max_no_days_files=max_no_days_files,
    max_no_days_lines=max_no_days_lines,
)

report["stats"]


In [ ]:
days_files = sorted(Path(paths.parsing_output).rglob(out_name))
len(days_files), days_files[:5]


In [ ]:
if days_files:
    sample_days = days_files[0]
    with open(sample_days, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        sample_rows = []
        for i, row in enumerate(reader):
            sample_rows.append(row)
            if i >= 10:
                break
    print(sample_days, sample_rows)
else:
    print("No days.csv files generated")


In [ ]:
files_without_days = report.get("files_without_days", [])
file_errors = report.get("file_errors", [])

{
    "report_json": str(report_json),
    "files_without_days_count": len(files_without_days),
    "file_errors_count": len(file_errors),
    "files_without_days_preview": files_without_days[:3],
    "file_errors_preview": file_errors[:3],
}
